##Imports

In [ ]:
#imports
%%capture
!pip install pymzml pyopenms pandas numpy tqdm
import pyopenms as oms
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
#gets mzML files from content
#chosen from https://www.ebi.ac.uk/pride/archive/projects/PXD017211

import os
mzml_files = [f for f in os.listdir() if f.endswith(".mzML")]
mzml_files

['181011_Leaders_11_4_E01.mzML',
 '181011_Leaders_6_4_E04.mzML',
 '181011_Leaders_34_7_E01.mzML']

In [ ]:
%%capture
!apt-get -qq install -y openms

##Examining Top Features

In [ ]:
import pyopenms as oms, numpy as np, pandas as pd, glob

def quick_prominent_features(mzml, max_ms1=500, top_k=30, mz_bin=0.01):
  exp = oms.MSExperiment()
  oms.MzMLFile().load(mzml, exp)

  bins = {}
  used = 0
  for spec in exp:
    if spec.getMSLevel()!=1 or spec.size()==0:
      continue
    mzs, ints = spec.get_peaks()
    mzs = np.array(mzs); ints = np.array(ints)
    b = np.round(mzs/mz_bin).astype(int)
    for bi,inten in zip(b,ints):
      if inten > bins.get(bi,0):
        bins[bi]=float(inten)
    used+=1
    if used>=max_ms1: break

  df = pd.DataFrame({"MZ":[k*mz_bin for k in bins], "MaxIntensity":list(bins.values())})
  return df.sort_values("MaxIntensity",ascending=False).head(top_k)

for f in sorted(glob.glob("*.mzML")):
  print("\n==",f,"==")
  display(quick_prominent_features(f, max_ms1=500, top_k=20))


== 181011_Leaders_11_4_E01.mzML ==


,MZ,MaxIntensity
224,401.92,664384.187500
128,378.95,484678.156250
25,354.89,444237.343750
285,416.88,383040.593750
211,399.91,314801.906250
685,580.79,297975.531250
1380,438.83,272605.250000
303,420.82,237125.781250
50,360.82,188524.234375
783,632.82,186772.062500



== 181011_Leaders_34_7_E01.mzML ==


,MZ,MaxIntensity
44,401.92,72677.468750
108,580.79,57264.179688
268,416.88,45082.558594
126,632.82,41849.062500
63,438.83,41436.292969
16,372.97,39000.097656
52,416.87,35578.421875
10,360.82,33386.890625
122,615.79,28240.595703
55,420.82,27943.363281



== 181011_Leaders_6_4_E04.mzML ==


,MZ,MaxIntensity
29,354.89,1.318249e+06
425,420.82,9.311674e+05
2158,416.87,8.769820e+05
494,438.83,7.582506e+05
403,416.88,7.224408e+05
186,378.95,7.063605e+05
68,360.82,6.867121e+05
1046,580.79,6.630595e+05
483,436.82,6.395928e+05
308,399.91,6.298863e+05


In [ ]:
import glob
import pandas as pd

mzmls = sorted(glob.glob("*.mzML"))

# Get top MS1 peaks per file
top_ms1 = {}
for f in mzmls:
  df = quick_prominent_features(f, max_ms1=500, top_k=50, mz_bin=0.01)  # adjust as needed
  top_ms1[f] = df

# Example
display(top_ms1[mzmls[0]].head(20))

,MZ,MaxIntensity
224,401.92,664384.187500
128,378.95,484678.156250
25,354.89,444237.343750
285,416.88,383040.593750
211,399.91,314801.906250
685,580.79,297975.531250
1380,438.83,272605.250000
303,420.82,237125.781250
50,360.82,188524.234375
783,632.82,186772.062500


##Proteomics Analysis Take #1

In [1]:
%%capture
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba create -y -n proteomics -c conda-forge -c bioconda msfragger philosopher

In [ ]:
!wget -q -O human_swissprot.fasta "https://rest.uniprot.org/uniprotkb/stream?query=organism_id:9606+AND+reviewed:true&format=fasta"
!ls -lh human_swissprot.fasta

-rw-r--r-- 1 root root 14M Jan 21 03:05 human_swissprot.fasta


In [ ]:
import glob
candidates = glob.glob("/root/.local/share/mamba/envs/proteomics/share/msfragger-*/MSFragger-*/MSFragger-*.jar")
msfragger_jar = sorted(candidates)[-1]
print("Using:", msfragger_jar)

Using: /root/.local/share/mamba/envs/proteomics/share/msfragger-4.2-0/MSFragger-4.2/MSFragger-4.2.jar


In [ ]:
params = """database_name = human_swissprot.fasta
num_threads = 4

# Use ppm tolerance (required to avoid precursor_mass_units=2 error)
precursor_mass_units = 1
precursor_mass_lower = -20
precursor_mass_upper = 20

fragment_mass_units = 1
fragment_mass_tolerance = 20

search_enzyme_name = Trypsin
allowed_missed_cleavage = 2
variable_mod_01 = 15.9949 M
output_report_topN = 1
"""
open("fragger.params","w").write(params)
!cat fragger.params

database_name = human_swissprot.fasta
num_threads = 4

# Use ppm tolerance (required to avoid precursor_mass_units=2 error)
precursor_mass_units = 1
precursor_mass_lower = -20
precursor_mass_upper = 20

fragment_mass_units = 1
fragment_mass_tolerance = 20

search_enzyme_name = Trypsin
allowed_missed_cleavage = 2
variable_mod_01 = 15.9949 M
output_report_topN = 1


In [ ]:
KEY = "5c6f23ef-cadb78c8-8cfcf03e-aea9baf6"
!./bin/micromamba run -n proteomics java -Xmx16G -jar "{msfragger_jar}" --key {KEY} fragger.params *.mzML

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
MSFragger version MSFragger-4.2
Batmass-IO version 1.35.1

timsdata library version timsdata-2-21-0-4
(c) University of Michigan
RawFileReader reading tool. Copyright (c) 2016 by Thermo Fisher Scientific, Inc. All rights reserved.
timdTOF .d reading tool. Copyright (c) 2022 by Bruker Daltonics GmbH & Co. KG. All rights reserved.
System OS: Linux, Architecture: amd64
Java Info: 25.0.1-internal, OpenJDK 64-Bit Server VM, Oracle Corporation
License key verified.
JVM started with 16 GB memory
Parameter 'data_type' was not supplied. Using default value: 0
Checking database...
Parameter 'num_enzyme_termini' was not supplied. Using default value: 2
Parameter 'search_enz

In [ ]:
!rm -rf .philosopher
!./bin/micromamba run -n proteomics philosopher workspace --init
!ls -la | head -n 50

time="03:08:07" level=info msg="Executing Workspace  v5.1.2"
time="03:08:08" level=info msg="Creating workspace"
time="03:08:08" level=info msg=Done
total 2624168
drwxr-xr-x 1 root root      4096 Jan 21 03:08 .
drwxr-xr-x 1 root root      4096 Jan 21 02:26 ..
-rw-r--r-- 1 root root 830684245 Jan 21 03:02 181011_Leaders_11_4_E01.mzML
-rw-r--r-- 1 root root  21985057 Jan 21 03:08 181011_Leaders_11_4_E01.pepXML
-rw-r--r-- 1 root root 841894108 Jan 21 03:02 181011_Leaders_34_7_E01.mzML
-rw-r--r-- 1 root root  25607761 Jan 21 03:08 181011_Leaders_34_7_E01.pepXML
-rw-r--r-- 1 root root 815947097 Jan 21 03:02 181011_Leaders_6_4_E04.mzML
-rw-r--r-- 1 root root  20793078 Jan 21 03:08 181011_Leaders_6_4_E04.pepXML
drwxr-xr-x 2 root root      4096 Jan 21 03:04 bin
drwxr-xr-x 4 root root      4096 Dec  9 14:41 .config
-rw-r--r-- 1 root root       365 Jan 21 03:05 fragger.params
-rw-r--r-- 1 root root  13665992 Jan 21 03:05 human_swissprot.fasta
-rw-r--r-- 1 root root  58253724 Jan 21 03:05 human_s

In [ ]:
!./bin/micromamba run -n proteomics philosopher database --annotate human_swissprot.fasta

time="03:08:08" level=info msg="Executing Database  v5.1.2"
time="03:08:08" level=info msg="Annotating the database"
time="03:08:08" level=info msg=Done


In [ ]:
!./bin/micromamba run -n proteomics philosopher peptideprophet --database human_swissprot.fasta *.pepXML

time="03:08:08" level=info msg="Executing PeptideProphet  v5.1.2"
 file 1: /content/181011_Leaders_11_4_E01.pepXML
No index list offset found. File will not be read.
No index list offset found. File will not be read.
 processed altogether 22906 results
INFO: Results written to file: /content/interact-181011_Leaders_11_4_E01.pep.xml
 file 1: /content/181011_Leaders_34_7_E01.pepXML
No index list offset found. File will not be read.
No index list offset found. File will not be read.
 processed altogether 26774 results
INFO: Results written to file: /content/interact-181011_Leaders_34_7_E01.pep.xml
 file 1: /content/181011_Leaders_6_4_E04.pepXML
No index list offset found. File will not be read.
No index list offset found. File will not be read.
 processed altogether 21808 results
INFO: Results written to file: /content/interact-181011_Leaders_6_4_E04.pep.xml

  - /content/interact-181011_Leaders_11_4_E01.pep.xml
  - Building Commentz-Walter keyword tree...
  - Searching the tree...
  - Li

In [ ]:
!./bin/micromamba run -n proteomics philosopher proteinprophet interact-*.pep.xml

time="03:08:29" level=info msg="Executing ProteinProphet  v5.1.2"
ProteinProphet (C++) by Insilicos LLC and LabKey Software, after the original Perl by A. Keller (TPP v6.0.0-rc15 Noctilucent, Build 202105021430-exported (Linux-x86_64))
 (no FPKM) (no groups) (using degen pep info)
Reading in /content/interact-181011_Leaders_11_4_E01.pep.xml...
...read in 0 1+, 6459 2+, 4755 3+, 0 4+, 0 5+, 0 6+, 0 7+ spectra with min prob 0.05

Reading in /content/interact-181011_Leaders_34_7_E01.pep.xml...
...read in 0 1+, 7425 2+, 5006 3+, 0 4+, 0 5+, 0 6+, 0 7+ spectra with min prob 0.05

Reading in /content/interact-181011_Leaders_6_4_E04.pep.xml...
...read in 0 1+, 6876 2+, 4696 3+, 0 4+, 0 5+, 0 6+, 0 7+ spectra with min prob 0.05

Initializing 14160 peptide weights: 0%...10%...20%...30%...40%...50%...60%...70%...80%...90%...100%
Calculating protein lengths and molecular weights from database /content/human_swissprot.fasta
.........:.........:.........:.........:.........:.........:.........:....

In [ ]:
!./bin/micromamba run -n proteomics philosopher filter \
  --pepxml interact-*.pep.xml \
  --psm 0.01 --pep 0.01 --prot 0.01

time="03:08:32" level=info msg="Executing Filter  v5.1.2"
time="03:08:32" level=info msg="Processing peptide identification files"
time="03:08:32" level=info msg="Parsing interact-181011_Leaders_11_4_E01.pep.xml"
time="03:08:32" level=info msg="1+ Charge profile" decoy=0 target=0
time="03:08:32" level=info msg="2+ Charge profile" decoy=0 target=6459
time="03:08:32" level=info msg="3+ Charge profile" decoy=0 target=4755
time="03:08:32" level=info msg="4+ Charge profile" decoy=0 target=0
time="03:08:32" level=info msg="5+ Charge profile" decoy=0 target=0
time="03:08:32" level=info msg="6+ Charge profile" decoy=0 target=0
time="03:08:32" level=info msg="Database search results" ions=5932 peptides=5507 psms=11214
time="03:08:32" level=info msg="Converged to 0.00 % FDR with 11214 PSMs" decoy=0 threshold=0.0501 total=11214
time="03:08:32" level=info msg="Converged to 0.00 % FDR with 5507 Peptides" decoy=0 threshold=0.0501 total=5507
time="03:08:32" level=info msg="Converged to 0.00 % FDR wit

In [ ]:
!./bin/micromamba run -n proteomics philosopher report
!ls -lh psm.tsv peptide.tsv protein.tsv

time="03:08:36" level=info msg="Executing Report  v5.1.2"
time="03:08:36" level=info msg="Creating reports"
time="03:08:36" level=info msg=Done
ls: cannot access 'protein.tsv': No such file or directory
-rw-r--r-- 1 root root 926K Jan 21 03:08 peptide.tsv
-rw-r--r-- 1 root root 4.4M Jan 21 03:08 psm.tsv


##Proteomics Analysis Take #2
Proteins.tsv was missing, so I had to retry the process

In [ ]:
from pathlib import Path

in_fa  = Path("human_swissprot.fasta")
out_fa = Path("human_swissprot_targetdecoy.fasta")

#reverses sequences to make decoy for false positives
def make_target_decoy_fasta(in_path, out_path, decoy_prefix="rev_"):
  with open(in_path) as fin, open(out_path, "w") as fout:
    header, seq = None, []
    for line in fin:
      line = line.rstrip("\n")
      if line.startswith(">"):
        if header is not None:
          s = "".join(seq)
          # write target
          fout.write(header + "\n")
          for i in range(0, len(s), 60):
            fout.write(s[i:i+60] + "\n")
          # write decoy (reverse sequence)
          decoy_header = ">" + decoy_prefix + header[1:]
          ds = s[::-1]
          fout.write(decoy_header + "\n")
          for i in range(0, len(ds), 60):
            fout.write(ds[i:i+60] + "\n")
        header, seq = line, []
      else:
        seq.append(line.strip())

    # last entry
    if header is not None:
      s = "".join(seq)
      fout.write(header + "\n")
      for i in range(0, len(s), 60):
        fout.write(s[i:i+60] + "\n")
      decoy_header = ">" + decoy_prefix + header[1:]
      ds = s[::-1]
      fout.write(decoy_header + "\n")
      for i in range(0, len(ds), 60):
        fout.write(ds[i:i+60] + "\n")

make_target_decoy_fasta(in_fa, out_fa)
print("Wrote:", out_fa, "size:", out_fa.stat().st_size/1e6, "MB")

Wrote: human_swissprot_targetdecoy.fasta size: 27.413664 MB


In [ ]:
params = open("fragger.params").read().splitlines()
params = [line for line in params if not line.strip().startswith("database_name")]
params.insert(0, "database_name = human_swissprot_targetdecoy.fasta")
open("fragger.params","w").write("\n".join(params) + "\n")
!head -n 20 fragger.params

database_name = human_swissprot_targetdecoy.fasta
num_threads = 4

# Use ppm tolerance (required to avoid precursor_mass_units=2 error)
precursor_mass_units = 1
precursor_mass_lower = -20
precursor_mass_upper = 20

fragment_mass_units = 1
fragment_mass_tolerance = 20

search_enzyme_name = Trypsin
allowed_missed_cleavage = 2
variable_mod_01 = 15.9949 M
output_report_topN = 1


In [ ]:
import glob
msfragger_jar = sorted(glob.glob("/root/.local/share/mamba/envs/proteomics/share/msfragger-*/MSFragger-*/MSFragger-*.jar"))[-1]
KEY = "5c6f23ef-cadb78c8-8cfcf03e-aea9baf6"

!rm -f *.pepXML interact-*.pep.xml 2>/dev/null || true
!./bin/micromamba run -n proteomics java -Xmx16G -jar "{msfragger_jar}" --key {KEY} fragger.params *.mzML
!ls -lh *.pepXML

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
MSFragger version MSFragger-4.2
Batmass-IO version 1.35.1

timsdata library version timsdata-2-21-0-4
(c) University of Michigan
RawFileReader reading tool. Copyright (c) 2016 by Thermo Fisher Scientific, Inc. All rights reserved.
timdTOF .d reading tool. Copyright (c) 2022 by Bruker Daltonics GmbH & Co. KG. All rights reserved.
System OS: Linux, Architecture: amd64
Java Info: 25.0.1-internal, OpenJDK 64-Bit Server VM, Oracle Corporation
License key verified.
JVM started with 16 GB memory
Parameter 'data_type' was not supplied. Using default value: 0
Checking database...
Parameter 'num_enzyme_termini' was not supplied. Using default value: 2
Parameter 'search_enz

In [ ]:
!./bin/micromamba run -n proteomics philosopher peptideprophet --database human_swissprot_targetdecoy.fasta *.pepXML

time="03:12:16" level=info msg="Executing PeptideProphet  v5.1.2"
 file 1: /content/181011_Leaders_11_4_E01.pepXML
No index list offset found. File will not be read.
No index list offset found. File will not be read.
 processed altogether 19563 results
INFO: Results written to file: /content/interact-181011_Leaders_11_4_E01.pep.xml
 file 1: /content/181011_Leaders_34_7_E01.pepXML
No index list offset found. File will not be read.
No index list offset found. File will not be read.
 processed altogether 23464 results
INFO: Results written to file: /content/interact-181011_Leaders_34_7_E01.pep.xml
 file 1: /content/181011_Leaders_6_4_E04.pepXML
No index list offset found. File will not be read.
No index list offset found. File will not be read.
 processed altogether 19054 results
INFO: Results written to file: /content/interact-181011_Leaders_6_4_E04.pep.xml

  - /content/interact-181011_Leaders_11_4_E01.pep.xml
  - Building Commentz-Walter keyword tree...
  - Searching the tree...
  - Li

In [ ]:
!./bin/micromamba run -n proteomics philosopher proteinprophet interact-*.pep.xml

time="03:12:36" level=info msg="Executing ProteinProphet  v5.1.2"
ProteinProphet (C++) by Insilicos LLC and LabKey Software, after the original Perl by A. Keller (TPP v6.0.0-rc15 Noctilucent, Build 202105021430-exported (Linux-x86_64))
 (no FPKM) (no groups) (using degen pep info)
Reading in /content/interact-181011_Leaders_11_4_E01.pep.xml...
...read in 0 1+, 4707 2+, 3114 3+, 0 4+, 0 5+, 0 6+, 0 7+ spectra with min prob 0.05

Reading in /content/interact-181011_Leaders_34_7_E01.pep.xml...
...read in 0 1+, 5905 2+, 3138 3+, 0 4+, 0 5+, 0 6+, 0 7+ spectra with min prob 0.05

Reading in /content/interact-181011_Leaders_6_4_E04.pep.xml...
...read in 0 1+, 5085 2+, 3177 3+, 0 4+, 0 5+, 0 6+, 0 7+ spectra with min prob 0.05

Initializing 6312 peptide weights: 0%...10%...20%...30%...40%...50%...60%...70%...80%...90%...100%
Calculating protein lengths and molecular weights from database /content/human_swissprot_targetdecoy.fasta
.........:.........:.........:.........:.........:.........:...

In [ ]:
!./bin/micromamba run -n proteomics philosopher filter --pepxml interact-*.pep.xml --psm 0.01 --pep 0.01 --prot 0.01
!./bin/micromamba run -n proteomics philosopher report
!ls -lh psm.tsv peptide.tsv protein.tsv

time="03:12:38" level=info msg="Executing Filter  v5.1.2"
time="03:12:38" level=info msg="Processing peptide identification files"
time="03:12:38" level=info msg="Parsing interact-181011_Leaders_11_4_E01.pep.xml"
time="03:12:39" level=info msg="1+ Charge profile" decoy=0 target=0
time="03:12:39" level=info msg="2+ Charge profile" decoy=0 target=4708
time="03:12:39" level=info msg="3+ Charge profile" decoy=0 target=3114
time="03:12:39" level=info msg="4+ Charge profile" decoy=0 target=0
time="03:12:39" level=info msg="5+ Charge profile" decoy=0 target=0
time="03:12:39" level=info msg="6+ Charge profile" decoy=0 target=0
time="03:12:39" level=info msg="Database search results" ions=3078 peptides=2702 psms=7822
time="03:12:39" level=info msg="Converged to 0.00 % FDR with 7821 PSMs" decoy=0 threshold=0.0501 total=7821
time="03:12:39" level=info msg="Converged to 0.00 % FDR with 2701 Peptides" decoy=0 threshold=0.0501 total=2701
time="03:12:39" level=info msg="Converged to 0.00 % FDR with 3

In [ ]:
!ls -lh *.prot.xml *.protXML 2>/dev/null || true
!ls -lh interact*.prot* 2>/dev/null || true

-rw-r--r-- 1 root root 4.7M Jan 21 03:12  interact.prot.xml
-rw-r--r-- 1 root root 4.7M Jan 21 03:12 interact.prot.xml


In [ ]:
!./bin/micromamba run -n proteomics philosopher filter \
  --pepxml interact-*.pep.xml \
  --protxml interact.prot.xml \
  --psm 0.01 --pep 0.01 --prot 0.01

time="03:12:42" level=info msg="Executing Filter  v5.1.2"
time="03:12:42" level=info msg="Processing peptide identification files"
time="03:12:42" level=info msg="Parsing interact-181011_Leaders_11_4_E01.pep.xml"
time="03:12:42" level=info msg="1+ Charge profile" decoy=0 target=0
time="03:12:42" level=info msg="2+ Charge profile" decoy=0 target=4708
time="03:12:42" level=info msg="3+ Charge profile" decoy=0 target=3114
time="03:12:42" level=info msg="4+ Charge profile" decoy=0 target=0
time="03:12:42" level=info msg="5+ Charge profile" decoy=0 target=0
time="03:12:42" level=info msg="6+ Charge profile" decoy=0 target=0
time="03:12:42" level=info msg="Database search results" ions=3078 peptides=2702 psms=7822
time="03:12:42" level=info msg="Converged to 0.00 % FDR with 7821 PSMs" decoy=0 threshold=0.0501 total=7821
time="03:12:42" level=info msg="Converged to 0.00 % FDR with 2701 Peptides" decoy=0 threshold=0.0501 total=2701
time="03:12:42" level=info msg="Converged to 0.00 % FDR with 3

In [ ]:
!./bin/micromamba run -n proteomics philosopher report
!ls -lh psm.tsv peptide.tsv protein.tsv

time="03:12:45" level=info msg="Executing Report  v5.1.2"
time="03:12:45" level=info msg="Creating reports"
time="03:12:45" level=info msg=Done
-rw-r--r-- 1 root root 335K Jan 21 03:12 peptide.tsv
-rw-r--r-- 1 root root 177K Jan 21 03:12 protein.tsv
-rw-r--r-- 1 root root 2.8M Jan 21 03:12 psm.tsv


##Proteomics Analysis

In [ ]:
import pandas as pd

#analyzes all proteins and their total count - sees which one has the greatest intensities
prot = pd.read_csv("protein.tsv", sep="\t")

id_col = next((c for c in prot.columns if "protein" in c.lower() or "accession" in c.lower()), prot.columns[0])
count_col = next((c for c in prot.columns if "psm" in c.lower() or "spectral" in c.lower()), None)

print("ID:", id_col)
print("COUNT:", count_col)

top = prot.sort_values(count_col, ascending=False).head(30)[[id_col, count_col]]
display(top)

ID: Protein
COUNT: Total Spectral Count


,Protein,Total Spectral Count
73,sp|P04114|APOB_HUMAN,647
30,sp|P01024|CO3_HUMAN,605
27,sp|P01011|AACT_HUMAN,405
143,sp|P19827|ITIH1_HUMAN,363
11,sp|P00450|CERU_HUMAN,297
25,sp|P01008|ANT3_HUMAN,264
113,sp|P0C0L4|CO4A_HUMAN,241
114,sp|P0C0L5|CO4B_HUMAN,232
208,sp|Q14624|ITIH4_HUMAN,219
52,sp|P02751|FINC_HUMAN,202


In [ ]:
import pandas as pd

#analyzes all peptides and their total count - sees which one has the greatest intensities
pep = pd.read_csv("peptide.tsv", sep="\t")

pep_col = next((c for c in pep.columns if "peptide" in c.lower() or "sequence" in c.lower()), pep.columns[0])
count_col = next((c for c in pep.columns if "psm" in c.lower() or "count" in c.lower()), None)

top_peptides = pep.sort_values(count_col, ascending=False).head(30)
display(top_peptides[[pep_col, count_col]])

,Peptide,Spectral Count
33,AFLEVNEEGSEAAASTAVVIAGR,186
827,HYYIGIIETTWDYASDHGEK,130
955,KDPEGLFLQDNIVAEFSVDETGQMSATAK,125
1226,LYGSEAFATDFQDSAAAK,111
146,AVLDVFEEGTEASAATAVK,107
1683,TAFISDFAVTADGNAFIGDIK,106
733,GTHVDLGLASANVDFAFSLYK,101
62,AIAANEADAVTLDAGLVYDAYLAPNNLKPVVAEFYGSK,90
192,DFSAEYEEDGKYEGLQEWEGK,72
486,EVVADSVWVDVK,67


In [ ]:
import pandas as pd
prot = pd.read_csv("protein.tsv", sep="\t")
prot.head()

,Protein,Protein ID,Entry Name,Gene,Length,Is Decoy,Is Contaminant,Organism,Protein Description,Protein Existence,...,Razor Peptides,Total Spectral Count,Unique Spectral Count,Razor Spectral Count,Total Intensity,Unique Intensity,Razor Intensity,Razor Assigned Modifications,Razor Observed Modifications,Indistinguishable Proteins
0,sp|A0A0C4DH38|HV551_HUMAN,A0A0C4DH38,HV551_HUMAN,IGHV5-51,117,False,False,Homo sapiens,Immunoglobulin heavy variable 5-51,1:Experimental evidence at protein level,...,0,5,5,0,0,0,0,NaN,NaN,NaN
1,sp|O00151|PDLI1_HUMAN,O00151,PDLI1_HUMAN,PDLIM1,329,False,False,Homo sapiens,PDZ and LIM domain protein 1,1:Experimental evidence at protein level,...,0,1,1,0,0,0,0,NaN,NaN,NaN
2,sp|O00391|QSOX1_HUMAN,O00391,QSOX1_HUMAN,QSOX1,747,False,False,Homo sapiens,Sulfhydryl oxidase 1,1:Experimental evidence at protein level,...,0,8,8,0,0,0,0,NaN,NaN,NaN
3,sp|O00533|NCHL1_HUMAN,O00533,NCHL1_HUMAN,CHL1,1208,False,False,Homo sapiens,Neural cell adhesion molecule L1-like protein,1:Experimental evidence at protein level,...,0,3,3,0,0,0,0,NaN,NaN,NaN
4,sp|O14786|NRP1_HUMAN,O14786,NRP1_HUMAN,NRP1,923,False,False,Homo sapiens,Neuropilin-1,1:Experimental evidence at protein level,...,0,6,6,0,0,0,0,NaN,NaN,NaN


In [ ]:
print(prot.columns.tolist())

['Protein', 'Protein ID', 'Entry Name', 'Gene', 'Length', 'Is Decoy', 'Is Contaminant', 'Organism', 'Protein Description', 'Protein Existence', 'Coverage', 'Protein Probability', 'Top Peptide Probability', 'Protein Qvalue', 'Total Peptides', 'Unique Peptides', 'Razor Peptides', 'Total Spectral Count', 'Unique Spectral Count', 'Razor Spectral Count', 'Total Intensity', 'Unique Intensity', 'Razor Intensity', 'Razor Assigned Modifications', 'Razor Observed Modifications', 'Indistinguishable Proteins']


In [ ]:
import pandas as pd

prot = pd.read_csv("protein.tsv", sep="\t")

# Keep only real human proteins, gets rid of decoys
prot_filt = prot[
  (prot["Is Decoy"] == False) &
  (prot["Is Contaminant"] == False) &
  (prot["Organism"].str.contains("Homo sapiens", na=False))
]

In [ ]:
prot_filt["ProminenceScore"] = (
  prot_filt["Unique Peptides"] * 2 +
  prot_filt["Unique Spectral Count"] +
  prot_filt["Unique Intensity"] / prot_filt["Unique Intensity"].max() * 10
)

In [ ]:
top15 = prot_filt.sort_values(by=["Protein Probability", "ProminenceScore", "Unique Peptides"], ascending=[False, False, False]).head(15)

top15[[
  "Protein ID", "Entry Name", "Gene",
  "Protein Description",
  "Unique Peptides", "Unique Spectral Count",
  "Unique Intensity", "Coverage",
  "Protein Probability", "Protein Qvalue",
  "ProminenceScore"
]]

,Protein ID,Entry Name,Gene,Protein Description,Unique Peptides,Unique Spectral Count,Unique Intensity,Coverage,Protein Probability,Protein Qvalue,ProminenceScore
73,P04114,APOB_HUMAN,APOB,Apolipoprotein B-100,123,647,0,35.00,1.0,0.0,NaN
30,P01024,CO3_HUMAN,C3,Complement C3,66,604,0,54.24,1.0,0.0,NaN
52,P02751,FINC_HUMAN,FN1,Fibronectin,45,202,0,30.16,1.0,0.0,NaN
208,Q14624,ITIH4_HUMAN,ITIH4,Inter-alpha-trypsin inhibitor heavy chain H4,34,216,0,48.06,1.0,0.0,NaN
11,P00450,CERU_HUMAN,CP,Ceruloplasmin,32,297,0,40.75,1.0,0.0,NaN
142,P19823,ITIH2_HUMAN,ITIH2,Inter-alpha-trypsin inhibitor heavy chain H2,29,148,0,41.12,1.0,0.0,NaN
31,P01031,CO5_HUMAN,C5,Complement C5,28,119,0,27.21,1.0,0.0,NaN
143,P19827,ITIH1_HUMAN,ITIH1,Inter-alpha-trypsin inhibitor heavy chain H1,26,363,0,42.48,1.0,0.0,NaN
29,P01023,A2MG_HUMAN,A2M,Alpha-2-macroglobulin,25,100,0,31.82,1.0,0.0,NaN
25,P01008,ANT3_HUMAN,SERPINC1,Antithrombin-III,19,264,0,44.18,1.0,0.0,NaN


In [ ]:
#top protein contributors to MDD
top15[["Gene", "Protein Description"]]

,Gene,Protein Description
73,APOB,Apolipoprotein B-100
30,C3,Complement C3
52,FN1,Fibronectin
208,ITIH4,Inter-alpha-trypsin inhibitor heavy chain H4
11,CP,Ceruloplasmin
142,ITIH2,Inter-alpha-trypsin inhibitor heavy chain H2
31,C5,Complement C5
143,ITIH1,Inter-alpha-trypsin inhibitor heavy chain H1
29,A2M,Alpha-2-macroglobulin
25,SERPINC1,Antithrombin-III
